In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

# 1. Carga de datos desde Drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/data/Rice_MSC_Dataset.csv')

# 2. Limpieza (Lógica de tu proyecto)
def limpiar_lote(df):
    for col in df.columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].mean())
    return df.drop_duplicates()

df = limpiar_lote(df)

# 3. Preparación Multi-clase
le = LabelEncoder()
df['CLASS_encoded'] = le.fit_transform(df['CLASS'])

X = df.drop(columns=['CLASS', 'CLASS_encoded', 'Calidad_Aceptable'], errors='ignore')
y = to_categorical(df['CLASS_encoded'])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

# 4. Modelo Multi-clase
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(y.shape[1], activation='softmax') # Softmax para múltiples clases
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=30, batch_size=32, verbose=0)

# 5. Generación de resultados: Matriz de Confusión
predicciones = model.predict(X_test)
clases_reales = np.argmax(y_test, axis=1)
clases_predichas = np.argmax(predicciones, axis=1)

cm = confusion_matrix(clases_reales, clases_predichas)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Matriz de Confusión: Clasificación por Variedad')
plt.ylabel('Real')
plt.xlabel('Predicho')
plt.show()
